In [2]:
import os
from pathlib import Path
import pandas as pd

%matplotlib inline

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)

    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p

    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path (the one containing data/raw): ").strip().strip('"')

    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    """Try OUT_DIR first; if the OS rejects the write, fall back to the home folder."""
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part1b_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print(f"Could not write to {OUT_DIR} ({e}).\nSaved to {fallback / filename} instead.")

# ---------- Data ----------
districts = pd.read_csv(RAW_DIR / "districts.csv")
age_sex = pd.read_csv(RAW_DIR / "age_sex.csv")

selected_districts = ["D05", "D24"]

# 1. Sum baseline 1 Jan 2025 population from age_sex.csv
pop25 = age_sex.groupby("district_id", as_index=False)["population_2025"].sum()

# 2. Reconstruct 1 Jan 2026 stock via balancing equation and compute 2025 rates
df_1b = districts[districts["district_id"].isin(selected_districts)].merge(pop25, on="district_id")
df_1b["net_migration_2025"] = df_1b["arrivals_2025"] - df_1b["departures_2025"]
df_1b["pop_2026_reconstructed"] = (
    df_1b["population_2025"]
    + df_1b["births_2025"]
    - df_1b["deaths_2025"]
    + df_1b["net_migration_2025"]
)
df_1b["diff_2026"] = df_1b["pop_2026_reconstructed"] - df_1b["population_2026"]

# Mid-year exposure as mean of 1 Jan 2025 and 1 Jan 2026 opening stocks
df_1b["mid_year_exposure_2025"] = 0.5 * (df_1b["population_2025"] + df_1b["population_2026"])

# Crude rates per 1,000 residents
df_1b["cbr_per_1000"] = (df_1b["births_2025"] / df_1b["mid_year_exposure_2025"]) * 1000.0
df_1b["cdr_per_1000"] = (df_1b["deaths_2025"] / df_1b["mid_year_exposure_2025"]) * 1000.0
df_1b["nmr_per_1000"] = (df_1b["net_migration_2025"] / df_1b["mid_year_exposure_2025"]) * 1000.0
df_1b["rni_per_1000"] = df_1b["cbr_per_1000"] - df_1b["cdr_per_1000"]
df_1b["total_growth_rate_per_1000"] = df_1b["rni_per_1000"] + df_1b["nmr_per_1000"]

export_cols = [
    "district_id", "population_2025", "births_2025", "deaths_2025",
    "arrivals_2025", "departures_2025", "net_migration_2025",
    "pop_2026_reconstructed", "population_2026", "diff_2026",
    "mid_year_exposure_2025", "cbr_per_1000", "cdr_per_1000",
    "nmr_per_1000", "rni_per_1000", "total_growth_rate_per_1000"
]

# Show the table first, then save
display(df_1b[export_cols])
safe_save(lambda p: df_1b[export_cols].to_csv(p, index=False), "part1b_balance_and_rates_d05_d24.csv")

BASE_DIR: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-


,district_id,population_2025,births_2025,deaths_2025,arrivals_2025,departures_2025,net_migration_2025,pop_2026_reconstructed,population_2026,diff_2026,mid_year_exposure_2025,cbr_per_1000,cdr_per_1000,nmr_per_1000,rni_per_1000,total_growth_rate_per_1000
0,D05,152444,3469,1597,3910,1749,2161,156477,156477,0,154460.5,22.458816,10.339213,13.990632,12.119603,26.110235
1,D24,172718,2197,3618,2841,3089,-248,171049,171049,0,171883.5,12.781913,21.049141,-1.442838,-8.267228,-9.710065


Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part1b_balance_and_rates_d05_d24.csv
